# LLaMA 2 layer superweight restoration plots

In [ ]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=2
%env HF_HUB_ENABLE_HF_TRANSFER=1
%config InlineBackend.close_figures = True

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import matplotlib as mpl
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

pd.set_option("display.max_colwidth", 260)

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / "src").exists() or (CWD / "emnlp").exists() else CWD.parent
OUT_DIR = REPO_ROOT / "emnlp" / "figs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = OUT_DIR / "superweight_layer_downproj_v2"
OUT_DIR.mkdir(exist_ok=True, parents=True)

MODEL_NAME = "meta-llama/Llama-2-7b-hf"
MODEL_ARCH_KEY = "llama"

LAYER_IDX = 1
MODULE_NAME = "mlp.down_proj"

# Known tracked superweights for this selected layer/module.
# Format: (row, col). Use [] if there are no known superweights for a layer.
SUPERWEIGHT_COORDS_LAYER = [(2533, 7890)] # []

# Hide inliers everywhere (scatter, restoration plot, CSV group curves) by setting this to False.
SHOW_INLIERS = True

TOPK = 128
INLIER_K = 128

# Inlier sampling range. This avoids near-zero inliers that make relative error unstable.
INLIER_MIN_ABS = 1e-4
INLIER_MIN_ABS_PERCENTILE = None  # e.g. 5.0, or None
INLIER_MAX_ABS_PERCENTILE = 50.0

BACKGROUND_SAMPLE = 5000

CALIB_NUM_SEQUENCES = 32
CALIB_SEQ_LEN = 256
CALIB_BATCH_SIZE = 1
CALIB_SEED = 42
FORCE_RECOMPUTE_DIAG = False

SVD_RANKS = [r for r in range(2, 4096, 32)]

# Restoration criterion for percent-restored curves:
# recovered iff abs_error <= max(abs_floor, rel_threshold * abs(original_weight)).
# so if rel_threshold=0.05, weight is considered as restored if its error after truncated svd and restoration is within 5% of original weight magnitude
RESTORE_REL_THRESHOLD = 0.05
RESTORE_ABS_FLOOR = 1e-6

# Restoration plot options.
THIRD_PANEL_MODE = "percent_restored"   # "percent_restored" or "relative_error"
REL_ERROR_AGG = "mean"                  # "mean" or "median"; used only for relative_error mode
RANK_PANEL_LAYOUT = "single"            # "single" or "lanes"
RECON_XSCALE = "linear"                 # "linear" or "log"
RECON_MARKER = None                     # None removes markers; e.g. "o" to show markers
RECON_MARKERSIZE = 0

# Combined figure options.
COMBINED_SHOW_TOP_LABELS = True
COMBINED_FIGSIZE = (12.2, 4.55)

# Marker options for the position-and-magnitude plot.
MARKER_ALPHA = {
    "background": 0.07,
    "inlier": 0.12,
    "top_magnitude": 0.88,
    "hassle_topk": 0.88,
    "superweight": 1.00,
}

# Tracked groups share one marker-size scale: marker area encodes |w|.
# Background is kept on a separate smaller scale by default so it does not dominate.
USE_SHARED_MARKER_SIZE_SCALE = True
BACKGROUND_USES_TRACKED_SIZE_SCALE = False

# Increase these to generally make tracked markers larger.
MARKER_SIZE_MIN = 6.0
MARKER_SIZE_MAX = 80.0

# Increase these to make the background sample larger.
BACKGROUND_SIZE_MIN = 1.0
BACKGROUND_SIZE_MAX = 6.0

# Size scaling. Sqrt scaling keeps high-magnitude outliers visible while avoiding oversized markers.
SIZE_SCALE = "sqrt"     # "log", "sqrt", or "linear"
SIZE_PMIN = 5.0
SIZE_PMAX = 99.7

GROUP_STYLES = {
    "inlier":        dict(color="#6E6E6E", marker=".", display="inliers"),
    "superweight":   dict(color="#EB5757", marker="o", display="superweights"),
    "top_magnitude": dict(color="#2F80ED", marker="o", display="top magnitude"),
    "hassle_topk":   dict(color="#27AE60", marker="D", display="activation-weighted importance top-k"),
}
GROUP_ORDER = ["inlier", "superweight", "top_magnitude", "hassle_topk"]

mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.size": 8.5,
    "axes.labelsize": 9.5,
    "axes.titlesize": 9.5,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.65,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "xtick.major.width": 0.65,
    "ytick.major.width": 0.65,
    "legend.fontsize": 7.0,
    "legend.frameon": True,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

print("Repo root:", REPO_ROOT)
print("Output dir:", OUT_DIR)

In [ ]:
@dataclass
class SuperweightConfig:
    model_name: str = MODEL_NAME
    model_arch_key: str = MODEL_ARCH_KEY
    model_dtype: Any = None
    device_map: Any = None
    decompose_device: str = None


def build_superweight_cfg():
    return SuperweightConfig(
        model_dtype=torch.float16,
        device_map="auto" if torch.cuda.is_available() else None,
        decompose_device="cuda" if torch.cuda.is_available() else "cpu",
    )


def sw_clean_mem():
    gc.collect()
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def load_superweight_model_and_tokenizer():
    from transformers import AutoModelForCausalLM, AutoTokenizer

    cfg = build_superweight_cfg()
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        torch_dtype=cfg.model_dtype,
        low_cpu_mem_usage=True,
        device_map=cfg.device_map,
    )
    model.eval()
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False
    return cfg, model, tokenizer


def get_module_by_name(root_module, dotted_name: str):
    module = root_module
    for part in dotted_name.split("."):
        if part.isdigit():
            module = module[int(part)]
        else:
            module = getattr(module, part)
    return module


def full_module_name_for_superweight(cfg, layer_idx: int, module_name: str) -> str:
    if cfg.model_arch_key != "llama":
        raise NotImplementedError(f"Unsupported model_arch_key: {cfg.model_arch_key}")
    return f"model.layers.{int(layer_idx)}.{module_name}"


def get_linear_module_for_analysis(model, cfg, layer_idx: int, module_name: str):
    full_name = full_module_name_for_superweight(cfg, layer_idx, module_name)
    module = get_module_by_name(model, full_name)
    if not isinstance(module, torch.nn.Linear):
        raise TypeError(f"{full_name} is {type(module).__name__}, expected nn.Linear")
    return module, full_name


def infer_input_device(model):
    try:
        return model.get_input_embeddings().weight.device
    except Exception:
        return next(model.parameters()).device


def build_wikitext2_train_loader(
    tokenizer,
    *,
    seq_len: int,
    batch_size: int,
    num_sequences: int,
    seed: int = 0,
    shuffle: bool = False,
):
    from torch.utils.data import DataLoader, TensorDataset

    from datasets import load_dataset
    ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    text = "\n\n".join(ds["text"])

    encoded = tokenizer(text, return_tensors="pt", add_special_tokens=False, truncation=False, verbose=False)
    ids = encoded.input_ids[0]
    n_blocks = ids.numel() // seq_len
    ids = ids[: n_blocks * seq_len].view(n_blocks, seq_len)

    subset_gen = torch.Generator().manual_seed(seed)
    loader_gen = torch.Generator().manual_seed(seed + 1)

    if ids.shape[0] > num_sequences:
        ids = ids[torch.randperm(ids.shape[0], generator=subset_gen)[:num_sequences]]

    return DataLoader(
        TensorDataset(ids),
        batch_size=batch_size,
        shuffle=shuffle,
        generator=loader_gen if shuffle else None,
    )


@torch.no_grad()
def collect_input_diag_scales(
    model,
    tokenizer,
    *,
    cfg,
    layer_idx: int,
    module_name: str,
    force_recompute: bool = False,
):
    cache_path = OUT_DIR / f"llama2_layer{int(layer_idx):02d}_{module_name.replace('.', '_')}_input_diag_scales.pt"
    if cache_path.exists() and not force_recompute:
        print(f"loading cached input diagonal scales → {cache_path}")
        return torch.load(cache_path, map_location="cpu"), cache_path

    module, _ = get_linear_module_for_analysis(model, cfg, layer_idx, module_name)
    stats = {"sumsq": None, "tokens": 0}

    def hook(_module, inputs, _outputs):
        x = inputs[0]
        if isinstance(x, tuple):
            x = x[0]
        x = x.detach()
        if x.ndim == 3:
            x = x.reshape(-1, x.shape[-1])
        elif x.ndim != 2:
            x = x.reshape(-1, x.shape[-1])
        x = x.to(device="cpu", dtype=torch.float32)
        sumsq = (x * x).sum(dim=0)
        if stats["sumsq"] is None:
            stats["sumsq"] = sumsq
        else:
            stats["sumsq"].add_(sumsq)
        stats["tokens"] += int(x.shape[0])

    handle = module.register_forward_hook(hook)
    loader = build_wikitext2_train_loader(
        tokenizer,
        seq_len=CALIB_SEQ_LEN,
        batch_size=CALIB_BATCH_SIZE,
        num_sequences=CALIB_NUM_SEQUENCES,
        seed=CALIB_SEED,
        shuffle=False,
    )
    device = infer_input_device(model)
    model.eval()
    try:
        for (input_ids,) in loader:
            _ = model(input_ids=input_ids.to(device), use_cache=False)
    finally:
        handle.remove()
        sw_clean_mem()

    diag = (stats["sumsq"] / max(1, int(stats["tokens"]))).clamp_min(1e-12).sqrt()
    torch.save(diag, cache_path)
    print(f"saved input diagonal scales → {cache_path}")
    return diag, cache_path

In [ ]:
def _coords_from_flat_indices(indices, n_cols: int) -> list[tuple[int, int]]:
    out = []
    for idx in indices:
        idx = int(idx)
        out.append((idx // n_cols, idx % n_cols))
    return out


def _active_group_order(df: pd.DataFrame | None = None) -> list[str]:
    if df is None or df.empty or "group" not in df.columns:
        present = set()
    else:
        present = set(df["group"].dropna().astype(str).tolist())

    out = []
    for group in GROUP_ORDER:
        if group == "inlier" and not SHOW_INLIERS:
            continue
        if group in present:
            out.append(group)
    return out


def sample_background_entries(W_cpu, *, n: int, seed: int, exclude_coords: set[tuple[int, int]]):
    rng = np.random.default_rng(seed)
    rows, cols = W_cpu.shape
    total = rows * cols
    picked = []
    seen = set(exclude_coords)
    max_attempts = max(1000, n * 20)
    attempts = 0

    while len(picked) < n and attempts < max_attempts:
        attempts += 1
        idx = int(rng.integers(0, total))
        r, c = divmod(idx, cols)
        key = (r, c)
        if key in seen:
            continue
        seen.add(key)
        val = float(W_cpu[r, c])
        picked.append({"row": r, "col": c, "value": val, "abs_value": abs(val)})

    return pd.DataFrame(picked)


def build_tracked_entries_df(
    W_cpu: np.ndarray,
    *,
    diag_scale: np.ndarray | None,
    top_k: int,
    inlier_k: int,
    seed: int = 0,
) -> pd.DataFrame:
    rows, cols = W_cpu.shape
    flat_abs = np.abs(W_cpu).reshape(-1)
    tracked_rows = []

    def add_group(group: str, coords: list[tuple[int, int]], score_name: str, scores: np.ndarray | None = None):
        for r, c in coords:
            if not (0 <= int(r) < rows and 0 <= int(c) < cols):
                print(f"[superweight warning] skipping out-of-range coord ({r}, {c}) for matrix shape {W_cpu.shape}")
                continue
            val = float(W_cpu[int(r), int(c)])
            item = {
                "group": group,
                "row": int(r),
                "col": int(c),
                "value": val,
                "abs_value": abs(val),
                "score_name": score_name,
            }
            if scores is not None:
                item["selection_score"] = float(scores[int(r), int(c)])
            else:
                item["selection_score"] = abs(val)
            tracked_rows.append(item)

    # Superweights. If COORDS_LAYER is empty, no superweight group is created.
    if len(SUPERWEIGHT_COORDS_LAYER):
        add_group("superweight", SUPERWEIGHT_COORDS_LAYER, "known_superweight")

    # Top-magnitude weights.
    k = min(int(top_k), flat_abs.size)
    if k > 0:
        top_idx = np.argpartition(-flat_abs, kth=k - 1)[:k]
        top_idx = top_idx[np.argsort(-flat_abs[top_idx])]
        add_group("top_magnitude", _coords_from_flat_indices(top_idx, cols), "abs_weight")

    # Activation-weighted top-k: abs(W_ij) * sqrt(E[x_j^2]).
    if diag_scale is None:
        diag = np.ones(cols, dtype=np.float32)
    else:
        diag = np.asarray(diag_scale, dtype=np.float32)
        if diag.shape[0] != cols:
            raise ValueError(f"diag_scale has length {diag.shape[0]}, expected input dimension {cols}")
    importance_scores = np.abs(W_cpu) * diag.reshape(1, -1)
    hflat = importance_scores.reshape(-1)
    hk = min(int(top_k), hflat.size)
    if hk > 0:
        importance_idx = np.argpartition(-hflat, kth=hk - 1)[:hk]
        importance_idx = importance_idx[np.argsort(-hflat[importance_idx])]
        add_group("hassle_topk", _coords_from_flat_indices(importance_idx, cols), "abs_weight_times_input_rms", importance_scores)

    if SHOW_INLIERS:
        # Random inliers from the configured magnitude interval, excluding selected groups.
        outlier_coords = {(int(x["row"]), int(x["col"])) for x in tracked_rows}
        upper_threshold = float(np.percentile(flat_abs, INLIER_MAX_ABS_PERCENTILE))

        if INLIER_MIN_ABS_PERCENTILE is not None:
            lower_threshold = max(
                float(INLIER_MIN_ABS),
                float(np.percentile(flat_abs, INLIER_MIN_ABS_PERCENTILE)),
            )
        else:
            lower_threshold = float(INLIER_MIN_ABS)

        candidate_flat = np.flatnonzero(
            (flat_abs >= lower_threshold) & (flat_abs <= upper_threshold)
        )

        if candidate_flat.size < inlier_k:
            print(
                f"[superweight warning] Only {candidate_flat.size} inlier candidates found "
                f"with |w| in [{lower_threshold:.3e}, {upper_threshold:.3e}]. "
                "Lower INLIER_MIN_ABS or increase INLIER_MAX_ABS_PERCENTILE."
            )

        rng = np.random.default_rng(seed)
        rng.shuffle(candidate_flat)

        inlier_coords = []
        for idx in candidate_flat:
            r, c = divmod(int(idx), cols)
            if (r, c) in outlier_coords:
                continue
            inlier_coords.append((r, c))
            if len(inlier_coords) >= inlier_k:
                break
        add_group("inlier", inlier_coords, f"random_abs_in_range_{lower_threshold:.1e}_to_p{INLIER_MAX_ABS_PERCENTILE:g}")

    df = pd.DataFrame(tracked_rows)
    if len(df):
        df["coord"] = list(zip(df["row"].astype(int), df["col"].astype(int)))
    return df


def compute_svd_entry_curves(W, entries_df: pd.DataFrame, rank_grid: list[int]):
    if entries_df.empty:
        raise ValueError("entries_df is empty. Nothing to analyze.")

    rows_t = torch.tensor(entries_df["row"].to_numpy(), device=W.device, dtype=torch.long)
    cols_t = torch.tensor(entries_df["col"].to_numpy(), device=W.device, dtype=torch.long)
    values_t = torch.tensor(entries_df["value"].to_numpy(), device=W.device, dtype=W.dtype)

    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    max_rank = int(S.numel())

    rank_grid = sorted({int(r) for r in rank_grid if int(r) >= 1 and int(r) <= max_rank})
    if max_rank not in rank_grid:
        rank_grid.append(max_rank)

    # Per-entry contribution from each singular component:
    # W_ij = sum_k U[i,k] * S[k] * Vh[k,j].
    U_rows = U.index_select(0, rows_t)
    V_cols = Vh[:, cols_t].transpose(0, 1).contiguous()
    contrib = U_rows * S.reshape(1, -1) * V_cols
    abs_contrib = contrib.abs()

    # Dominant singular component for each tracked entry.
    dominant_idx = abs_contrib.argmax(dim=1)
    dominant_rank = dominant_idx + 1
    dominant_singular_value = S.index_select(0, dominant_idx)
    dominant_abs_contrib = abs_contrib.gather(1, dominant_idx.reshape(-1, 1)).squeeze(1)
    abs_contrib_sum = abs_contrib.sum(dim=1).clamp_min(1e-12)
    dominant_share = dominant_abs_contrib / abs_contrib_sum

    singular_alignment = entries_df[["group", "row", "col", "value", "abs_value"]].copy()
    singular_alignment["dominant_rank"] = dominant_rank.detach().cpu().numpy().astype(int)
    singular_alignment["dominant_rank_pct"] = (dominant_rank.float() / float(max_rank)).detach().cpu().numpy()
    singular_alignment["dominant_singular_value"] = dominant_singular_value.detach().cpu().float().numpy()
    singular_alignment["dominant_abs_contribution"] = dominant_abs_contrib.detach().cpu().float().numpy()
    singular_alignment["dominant_share_abs_contribution"] = dominant_share.detach().cpu().float().numpy()

    # Cumulative absolute contribution mass by rank.
    prefix_abs_mass = torch.cumsum(abs_contrib, dim=1) / abs_contrib_sum.reshape(-1, 1)

    entry_curve_rows = []
    mass_rows = []

    for rank in rank_grid:
        rank = int(rank)
        recon = contrib[:, :rank].sum(dim=1)
        abs_error = (recon - values_t).abs()
        rel_error = abs_error / values_t.abs().clamp_min(1e-12)
        recovered = abs_error <= torch.maximum(
            torch.full_like(abs_error, float(RESTORE_ABS_FLOOR)),
            float(RESTORE_REL_THRESHOLD) * values_t.abs(),
        )

        tmp = entries_df[["group", "row", "col", "value", "abs_value"]].copy()
        tmp["rank"] = rank
        tmp["reconstructed_value"] = recon.detach().cpu().float().numpy()
        tmp["abs_error"] = abs_error.detach().cpu().float().numpy()
        tmp["rel_error"] = rel_error.detach().cpu().float().numpy()
        tmp["recovered"] = recovered.detach().cpu().numpy().astype(bool)
        entry_curve_rows.append(tmp)

        mtmp = entries_df[["group", "row", "col", "value", "abs_value"]].copy()
        mtmp["rank"] = rank
        mtmp["cum_abs_contrib_share"] = prefix_abs_mass[:, rank - 1].detach().cpu().float().numpy()
        mass_rows.append(mtmp)

    entry_curves = pd.concat(entry_curve_rows, ignore_index=True)
    entry_mass_curves = pd.concat(mass_rows, ignore_index=True)

    group_rows = []
    for (group, rank), g in entry_curves.groupby(["group", "rank"]):
        group_rows.append({
            "group": group,
            "rank": int(rank),
            "n_entries": int(len(g)),
            "restored_pct": float(g["recovered"].mean() * 100.0),
            "mean_rel_error": float(g["rel_error"].mean()),
            "median_rel_error": float(g["rel_error"].median()),
            "mean_abs_error": float(g["abs_error"].mean()),
            "median_abs_error": float(g["abs_error"].median()),
        })
    group_curves = pd.DataFrame(group_rows)

    mass_group_rows = []
    for (group, rank), g in entry_mass_curves.groupby(["group", "rank"]):
        mass_group_rows.append({
            "group": group,
            "rank": int(rank),
            "n_entries": int(len(g)),
            "mean_cum_abs_contrib_share": float(g["cum_abs_contrib_share"].mean()),
            "median_cum_abs_contrib_share": float(g["cum_abs_contrib_share"].median()),
        })
    singular_mass_curves = pd.DataFrame(mass_group_rows)

    singular_values = pd.DataFrame({
        "rank_index_1based": np.arange(1, max_rank + 1),
        "singular_value": S.detach().cpu().float().numpy(),
    })

    return entry_curves, group_curves, singular_values, singular_alignment, singular_mass_curves

In [ ]:
def _style_legend_frame(legend):
    if legend is None:
        return None
    legend.get_frame().set_facecolor("#FFFBFB")
    legend.get_frame().set_edgecolor("#D0D0D0")
    legend.get_frame().set_linewidth(0.6)
    legend.get_frame().set_alpha(1.0)
    return legend


def _style_plain_axis(ax, *, grid: bool = True):
    ax.set_facecolor("white")
    if grid:
        ax.grid(True, linestyle=(0, (3, 2)), linewidth=0.45, color="#D8CFC2", alpha=0.78)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#B9B1A8")
        spine.set_linewidth(0.7)
    ax.tick_params(axis="both", which="major", color="#9A8F84", labelcolor="0.15")


def save_figure_pdf_png(fig, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    for ext in [".pdf", ".png"]:
        p = out_path.with_suffix(ext)
        fig.savefig(p, bbox_inches="tight", pad_inches=0.04)
        print(f"saved → {p}")


def scale_marker_sizes(values, *, s_min: float, s_max: float, vmin: float, vmax: float):
    values = np.asarray(values, dtype=float)
    values = np.maximum(values, 0.0)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        return np.full_like(values, (s_min + s_max) / 2.0, dtype=float)
    if SIZE_SCALE == "log":
        x = np.log1p(values)
        lo = np.log1p(max(vmin, 0.0))
        hi = np.log1p(max(vmax, 0.0))
        z = np.clip((x - lo) / max(hi - lo, 1e-12), 0.0, 1.0)
    elif SIZE_SCALE == "sqrt":
        z = np.sqrt(np.clip((values - vmin) / max(vmax - vmin, 1e-12), 0.0, 1.0))
    else:
        z = np.clip((values - vmin) / max(vmax - vmin, 1e-12), 0.0, 1.0)
    return s_min + (s_max - s_min) * z


def _marker_scale_limits(entries_df: pd.DataFrame, background_df: pd.DataFrame):
    all_sizes = []
    if len(entries_df):
        all_sizes.extend(entries_df["abs_value"].to_numpy())
    if not all_sizes and len(background_df):
        all_sizes.extend(background_df["abs_value"].to_numpy())
    all_sizes = np.asarray(all_sizes, dtype=float)
    if len(all_sizes):
        return float(np.nanpercentile(all_sizes, SIZE_PMIN)), float(np.nanpercentile(all_sizes, SIZE_PMAX))
    return 0.0, 1.0


def _background_marker_scale_limits(background_df: pd.DataFrame):
    if not len(background_df):
        return 0.0, 1.0
    vals = background_df["abs_value"].to_numpy(dtype=float)
    return float(np.nanpercentile(vals, 5.0)), float(np.nanpercentile(vals, 99.5))


def draw_weight_map_panel(ax, *, entries_df: pd.DataFrame, background_df: pd.DataFrame, matrix_shape: tuple[int, int], title: str, show_legend: bool = True):
    vmin, vmax = _marker_scale_limits(entries_df, background_df)

    if len(background_df):
        if BACKGROUND_USES_TRACKED_SIZE_SCALE:
            bg_s = scale_marker_sizes(
                background_df["abs_value"].to_numpy(),
                s_min=MARKER_SIZE_MIN,
                s_max=MARKER_SIZE_MAX,
                vmin=vmin,
                vmax=vmax,
            )
        else:
            bg_vmin, bg_vmax = _background_marker_scale_limits(background_df)
            bg_s = scale_marker_sizes(
                background_df["abs_value"].to_numpy(),
                s_min=BACKGROUND_SIZE_MIN,
                s_max=BACKGROUND_SIZE_MAX,
                vmin=bg_vmin,
                vmax=bg_vmax,
            )
        ax.scatter(
            background_df["col"], background_df["row"],
            s=bg_s,
            color="#6E6E6E",
            alpha=MARKER_ALPHA["background"],
            linewidths=0,
            zorder=1,
            label="background sample",
        )

    # All tracked groups use the same abs(weight) size scale.
    for group in _active_group_order(entries_df):
        g = entries_df[entries_df["group"] == group]
        if g.empty:
            continue
        style = GROUP_STYLES[group]
        s = scale_marker_sizes(
            g["abs_value"].to_numpy(),
            s_min=MARKER_SIZE_MIN,
            s_max=MARKER_SIZE_MAX,
            vmin=vmin,
            vmax=vmax,
        )
        zorder = 2 if group == "inlier" else (5 if group == "superweight" else 4)
        edge = "none" if style["marker"] == "." else "0.15"
        lw = 0 if style["marker"] == "." else 0.40
        ax.scatter(
            g["col"], g["row"],
            s=s,
            marker=style["marker"],
            color=style["color"],
            edgecolors=edge,
            linewidths=lw,
            alpha=MARKER_ALPHA.get(group, 0.85),
            zorder=zorder,
            label=f"{style['display']} (n={len(g)})",
        )

    ax.set_title(title, fontsize=9.5, fontweight="semibold", pad=4)
    ax.set_xlabel("Column (input dimension)")
    ax.set_ylabel("Row (output dimension)")
    ax.set_xlim(-0.02 * matrix_shape[1], 1.02 * matrix_shape[1])
    ax.set_ylim(-0.02 * matrix_shape[0], 1.02 * matrix_shape[0])
    _style_plain_axis(ax, grid=True)

    if show_legend:
        legend = ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.20), ncol=2, frameon=True, fancybox=False, fontsize=6.5)
        _style_legend_frame(legend)


def _rank_panel_y_values(g: pd.DataFrame, *, mode: str, rel_error_agg: str):
    if mode == "percent_restored":
        return g["restored_pct"].to_numpy(), "% restored"
    if mode == "relative_error":
        col = f"{rel_error_agg}_rel_error"
        if col not in g.columns:
            raise ValueError(f"Unknown rel_error_agg={rel_error_agg!r}; expected 'mean' or 'median'.")
        return g[col].to_numpy(), f"{rel_error_agg} relative error"
    raise ValueError("mode must be 'percent_restored' or 'relative_error'")


def draw_rank_recovery_single_axis(ax, *, group_curves: pd.DataFrame, mode: str, rel_error_agg: str = "median", title: str, show_legend: bool = True):
    y_all = []
    for group in _active_group_order(group_curves):
        g = group_curves[group_curves["group"] == group].sort_values("rank")
        if g.empty:
            continue
        style = GROUP_STYLES[group]
        y, y_label = _rank_panel_y_values(g, mode=mode, rel_error_agg=rel_error_agg)
        y_all.extend(list(y))
        ax.plot(
            g["rank"], y,
            color=style["color"],
            marker=RECON_MARKER,
            linewidth=1.25,
            markersize=RECON_MARKERSIZE,
            label=style["display"],
            alpha=0.95,
        )

    ax.set_xscale(RECON_XSCALE)
    ax.set_xlabel("SVD rank kept")

    if mode == "percent_restored":
        ax.set_ylabel("Tracked entries restored (%)")
        ax.set_ylim(-3, 103)
        ax.axhline(100, color="#9A8F84", linestyle=":", linewidth=0.75, alpha=0.8)
    else:
        ax.set_ylabel(y_label)
        if y_all:
            ymin, ymax = float(np.nanmin(y_all)), float(np.nanmax(y_all))
            pad = 0.08 * max(ymax - ymin, 1e-9)
            ax.set_ylim(max(0, ymin - pad), ymax + pad)

    ax.set_title(title, fontsize=9.5, fontweight="semibold", pad=4)
    _style_plain_axis(ax, grid=True)

    if show_legend:
        legend = ax.legend(loc="best", frameon=True, fancybox=False, fontsize=6.4)
        _style_legend_frame(legend)
    return [ax]


def draw_rank_recovery_lanes(fig, outer_spec, *, group_curves: pd.DataFrame, mode: str, rel_error_agg: str = "median", title: str):
    groups = _active_group_order(group_curves)
    if not groups:
        ax = fig.add_subplot(outer_spec)
        ax.text(0.5, 0.5, "No tracked groups", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return [ax]

    sub = outer_spec.subgridspec(len(groups), 1, hspace=0.10)
    axes = []
    for i, group in enumerate(groups):
        ax = fig.add_subplot(sub[i, 0], sharex=axes[0] if axes else None)
        axes.append(ax)
        g = group_curves[group_curves["group"] == group].sort_values("rank")
        style = GROUP_STYLES[group]
        if len(g):
            y, _ = _rank_panel_y_values(g, mode=mode, rel_error_agg=rel_error_agg)
            if mode == "percent_restored":
                ax.set_ylim(-3, 103)
                ax.axhline(100, color="#9A8F84", linestyle=":", linewidth=0.75, alpha=0.8)
            else:
                ymin, ymax = float(np.nanmin(y)), float(np.nanmax(y))
                pad = 0.08 * max(ymax - ymin, 1e-9)
                ax.set_ylim(max(0, ymin - pad), ymax + pad)
            ax.plot(
                g["rank"], y,
                color=style["color"],
                marker=RECON_MARKER,
                linewidth=1.2,
                markersize=RECON_MARKERSIZE,
            )
        ax.text(
            0.01, 0.82, style["display"],
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=6.6,
            bbox=dict(boxstyle="round,pad=0.16", fc="white", ec="none", alpha=0.85),
        )
        ax.set_xscale(RECON_XSCALE)
        _style_plain_axis(ax, grid=True)
        if i < len(groups) - 1:
            plt.setp(ax.get_xticklabels(), visible=False)
        else:
            ax.set_xlabel("SVD rank kept")
        ax.set_ylabel("%" if mode == "percent_restored" else "rel. err", labelpad=2)

    axes[0].set_title(title, fontsize=9.5, fontweight="semibold", pad=4)
    return axes


def draw_rank_recovery_panel(fig, outer_spec, *, group_curves: pd.DataFrame, mode: str, rel_error_agg: str = "median", title: str, layout: str = "single", show_legend: bool = True):
    if layout == "single":
        ax = fig.add_subplot(outer_spec)
        return draw_rank_recovery_single_axis(
            ax,
            group_curves=group_curves,
            mode=mode,
            rel_error_agg=rel_error_agg,
            title=title,
            show_legend=show_legend,
        )
    if layout == "lanes":
        return draw_rank_recovery_lanes(
            fig,
            outer_spec,
            group_curves=group_curves,
            mode=mode,
            rel_error_agg=rel_error_agg,
            title=title,
        )
    raise ValueError("layout must be 'single' or 'lanes'")


def plot_superweight_weight_map(entries_df, background_df, matrix_shape, *, out_path: Path, show: bool = True):
    fig, ax = plt.subplots(1, 1, figsize=(6.2, 4.3))
    fig.patch.set_facecolor("white")
    fig.subplots_adjust(left=0.12, right=0.98, top=0.88, bottom=0.29)
    draw_weight_map_panel(
        ax,
        entries_df=entries_df,
        background_df=background_df,
        matrix_shape=matrix_shape,
        title=f"Layer {LAYER_IDX} · {MODULE_NAME}: position and magnitude",
        show_legend=True,
    )
    save_figure_pdf_png(fig, out_path)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


def plot_superweight_rank_panel(group_curves, *, mode: str, rel_error_agg: str, layout: str, out_path: Path, show: bool = True):
    figsize = (5.8, 4.35) if layout == "single" else (5.6, 5.6)
    fig = plt.figure(figsize=figsize)
    fig.patch.set_facecolor("white")
    fig.subplots_adjust(left=0.13, right=0.98, top=0.92, bottom=0.13)
    title = "Recovery under truncated SVD" if mode == "percent_restored" else f"{rel_error_agg.capitalize()} relative error under truncated SVD"
    gs = fig.add_gridspec(1, 1)
    draw_rank_recovery_panel(
        fig,
        gs[0, 0],
        group_curves=group_curves,
        mode=mode,
        rel_error_agg=rel_error_agg,
        title=title,
        layout=layout,
        show_legend=True,
    )
    save_figure_pdf_png(fig, out_path)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


def _combined_handles(entries_df: pd.DataFrame, include_background: bool = True):
    handles = []
    for group in _active_group_order(entries_df):
        style = GROUP_STYLES[group]
        handles.append(
            Line2D(
                [], [], marker=style["marker"], linestyle="None",
                color=style["color"], markeredgecolor="0.15",
                label=style["display"],
            )
        )
    if include_background and len(handles):
        handles.append(
            Line2D(
                [], [], marker="o", linestyle="None",
                color="#6E6E6E",
                alpha=MARKER_ALPHA["background"],
                label="background sample",
            )
        )
    return handles


def plot_superweight_combined_two_panel(
    entries_df,
    background_df,
    group_curves,
    matrix_shape,
    *,
    mode: str,
    rel_error_agg: str,
    rank_layout: str,
    show_top_labels: bool,
    out_path: Path,
    show: bool = True,
):
    fig = plt.figure(figsize=COMBINED_FIGSIZE)
    fig.patch.set_facecolor("white")

    # One horizontal row: left = position/magnitude, right = restoration.
    gs = fig.add_gridspec(
        1, 2,
        width_ratios=[1.10, 1.10],
        left=0.065,
        right=0.985,
        top=0.88 if show_top_labels else 0.94,
        bottom=0.20,
        wspace=0.16,
    )

    ax_map = fig.add_subplot(gs[0, 0])
    map_title = "(a) position and magnitude" if show_top_labels else ""
    draw_weight_map_panel(
        ax_map,
        entries_df=entries_df,
        background_df=background_df,
        matrix_shape=matrix_shape,
        title=map_title,
        show_legend=False,
    )

    rank_title = "(b) recovery under truncated SVD" if mode == "percent_restored" else f"(b) {rel_error_agg} relative error under truncated SVD"
    if not show_top_labels:
        rank_title = ""
    draw_rank_recovery_panel(
        fig,
        gs[0, 1],
        group_curves=group_curves,
        mode=mode,
        rel_error_agg=rel_error_agg,
        title=rank_title,
        layout=rank_layout,
        show_legend=False,
    )

    handles = _combined_handles(entries_df, include_background=True)
    if handles:
        legend = fig.legend(
            handles=handles,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.075),
            ncol=min(5, len(handles)),
            frameon=True,
            fancybox=False,
            fontsize=6.7,
        )
        _style_legend_frame(legend)

    if show_top_labels:
        fig.suptitle(
            f"LLaMA 2 7B · layer {LAYER_IDX} · {MODULE_NAME}",
            fontsize=10.5,
            fontweight="semibold",
            y=0.98,
        )

    save_figure_pdf_png(fig, out_path)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


In [ ]:
# Run selected-layer analysis and save CSV files

CFG, MODEL, TOKENIZER = load_superweight_model_and_tokenizer()

DIAG_SCALES, DIAG_CACHE_PATH = collect_input_diag_scales(
    MODEL,
    TOKENIZER,
    cfg=CFG,
    layer_idx=LAYER_IDX,
    module_name=MODULE_NAME,
    force_recompute=FORCE_RECOMPUTE_DIAG,
)

module, full_name = get_linear_module_for_analysis(
    MODEL,
    CFG,
    LAYER_IDX,
    MODULE_NAME,
)

work_device = torch.device(CFG.decompose_device if torch.cuda.is_available() else module.weight.device)
W = module.weight.detach().to(device=work_device, dtype=torch.float32).contiguous()
W_cpu = W.detach().cpu().numpy().astype(np.float32)

entries_df = build_tracked_entries_df(
    W_cpu,
    diag_scale=DIAG_SCALES.detach().cpu().numpy(),
    top_k=TOPK,
    inlier_k=INLIER_K,
    seed=LAYER_IDX,
)

exclude_coords = {(int(r), int(c)) for r, c in zip(entries_df["row"], entries_df["col"])}
background_df = sample_background_entries(
    W_cpu,
    n=BACKGROUND_SAMPLE,
    seed=LAYER_IDX,
    exclude_coords=exclude_coords,
)

(
    entry_curves_df,
    group_curves_df,
    singular_values_df,
    singular_alignment_df,
    singular_mass_curves_df,
) = compute_svd_entry_curves(
    W,
    entries_df,
    SVD_RANKS,
)

stem = f"llama2_layer{LAYER_IDX:02d}_{MODULE_NAME.replace('.', '_')}"

entries_path = OUT_DIR / f"{stem}_tracked_entries.csv"
background_path = OUT_DIR / f"{stem}_background_sample.csv"
entry_curves_path = OUT_DIR / f"{stem}_entry_reconstruction_curves.csv"
group_curves_path = OUT_DIR / f"{stem}_group_reconstruction_curves.csv"
singular_values_path = OUT_DIR / f"{stem}_singular_values.csv"
singular_alignment_path = OUT_DIR / f"{stem}_singular_alignment.csv"
singular_mass_curves_path = OUT_DIR / f"{stem}_singular_contribution_mass_curves.csv"

entries_df.to_csv(entries_path, index=False)
background_df.to_csv(background_path, index=False)
entry_curves_df.to_csv(entry_curves_path, index=False)
group_curves_df.to_csv(group_curves_path, index=False)
singular_values_df.to_csv(singular_values_path, index=False)
singular_alignment_df.to_csv(singular_alignment_path, index=False)
singular_mass_curves_df.to_csv(singular_mass_curves_path, index=False)

for p in [
    entries_path,
    background_path,
    entry_curves_path,
    group_curves_path,
    singular_values_path,
    singular_alignment_path,
    singular_mass_curves_path,
]:
    print(f"saved → {p}")

print("Tracked groups:", _active_group_order(entries_df))
display(entries_df.sort_values(["group", "abs_value"], ascending=[True, False]).head(80))
display(group_curves_df.head(20))
display(singular_alignment_df.sort_values(["group", "dominant_rank"]).head(40))


In [ ]:
plot_superweight_weight_map(
    entries_df,
    background_df,
    tuple(W_cpu.shape),
    out_path=OUT_DIR / f"{stem}_weight_map.pdf",
)

plot_superweight_rank_panel(
    group_curves_df,
    mode=THIRD_PANEL_MODE,
    rel_error_agg=REL_ERROR_AGG,
    layout=RANK_PANEL_LAYOUT,
    out_path=OUT_DIR / (
        f"{stem}_restoration_"
        f"{RANK_PANEL_LAYOUT}_"
        f"{THIRD_PANEL_MODE}.pdf"
    ),
)

plot_superweight_combined_two_panel(
    entries_df,
    background_df,
    group_curves_df,
    tuple(W_cpu.shape),
    mode=THIRD_PANEL_MODE,
    rel_error_agg=REL_ERROR_AGG,
    rank_layout=RANK_PANEL_LAYOUT,
    show_top_labels=COMBINED_SHOW_TOP_LABELS,
    out_path=OUT_DIR / (
        f"{stem}_combined_"
        f"{RANK_PANEL_LAYOUT}_"
        f"{THIRD_PANEL_MODE}.pdf"
    ),
)

# del MODEL, TOKENIZER, W
# sw_clean_mem()